# 03 — Análisis Descriptivo
**AgroVision AI** · Estadísticas y correlaciones

## 1. Cargar datos limpios

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False, 'axes.spines.right': False})

try:
    df = pd.read_csv('data_limpio.csv')
    print(f"Cargado desde CSV: {df.shape}")
except:
    import requests
    URL = "https://www.datos.gov.co/resource/uejq-wxrr.json"
    df = pd.DataFrame(requests.get(URL, params={"$limit": 5000, "$where": "rendimiento IS NOT NULL AND rendimiento > 0"}, timeout=60).json())
    for col in ['area_sembrada','area_cosechada','produccion','rendimiento','anio']:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')
    df.rename(columns={'a_o':'anio','rea_sembrada':'area_sembrada','producci_n':'produccion','rea_cosechada':'area_cosechada'}, inplace=True)
    df = df[df['rendimiento'].between(0.01, 50)].dropna(subset=['area_sembrada'])
    print(f"Cargado desde API: {df.shape}")

## 2. Estadísticas descriptivas

In [ ]:
print("=== Estadísticas del rendimiento (t/ha) ===")
print(df['rendimiento'].describe())
print(f"\nCoeficiente de variación: {df['rendimiento'].std()/df['rendimiento'].mean()*100:.1f}%")

## 3. Top 10 cultivos por rendimiento promedio

In [ ]:
top_rend = df.groupby('cultivo')['rendimiento'].agg(['mean','std','count']).round(2)
top_rend.columns = ['rendimiento_prom','std','registros']
top_rend = top_rend[top_rend['registros'] >= 50].sort_values('rendimiento_prom', ascending=False).head(10)

fig, ax = plt.subplots()
top_rend['rendimiento_prom'].plot(kind='bar', ax=ax, color='#2D6A4F', yerr=top_rend['std'])
ax.set_title('Top 10 cultivos por rendimiento promedio (t/ha)')
ax.set_xlabel('Cultivo')
ax.set_ylabel('Rendimiento (t/ha)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('top_cultivos_rendimiento.png', dpi=150, bbox_inches='tight')
plt.show()
print(top_rend)

## 4. Rendimiento por departamento

In [ ]:
rend_depto = df.groupby('departamento')['rendimiento'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
rend_depto.plot(kind='barh', ax=ax, color='#52B788')
ax.set_title('Rendimiento promedio por departamento (t/ha)')
ax.set_xlabel('Rendimiento promedio (t/ha)')
plt.tight_layout()
plt.savefig('rendimiento_por_departamento.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Matriz de correlaciones

In [ ]:
num_cols = ['area_sembrada','area_cosechada','rendimiento','anio']
num_cols = [c for c in num_cols if c in df.columns]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=10)
ax.set_title('Matriz de correlaciones — Variables numéricas')
plt.tight_layout()
plt.savefig('correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print(corr)

## 6. Evolución temporal del rendimiento

In [ ]:
if 'anio' in df.columns:
    evo = df.groupby('anio')['rendimiento'].agg(['mean','std']).reset_index()
    fig, ax = plt.subplots()
    ax.plot(evo['anio'], evo['mean'], 'o-', color='#1B4332', linewidth=2)
    ax.fill_between(evo['anio'], evo['mean']-evo['std'], evo['mean']+evo['std'], alpha=0.2, color='#52B788')
    ax.set_title('Evolución del rendimiento promedio nacional 2019-2024')
    ax.set_xlabel('Año')
    ax.set_ylabel('Rendimiento promedio (t/ha)')
    plt.tight_layout()
    plt.savefig('evolucion_temporal.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(evo)

## 7. Conclusiones

- Alta variabilidad entre cultivos (CV > 100%)
- Boyacá y Cundinamarca lideran en volumen de registros
- Correlación baja entre área sembrada y rendimiento — el tipo de cultivo es más determinante
- El rendimiento promedio nacional muestra estabilidad entre 2019-2023 con caída en 2024 (datos incompletos)